# Get To Know A Dataset: OncoGenerator-Endo

**OncoGenerator-Endo** is a *synthetic* endometrial-cancer cohort: clinical variables plus log2 fold-change values for 180 proteins, generated with a known, recorded ground truth. Because the true effects are known, the data are meant for **benchmarking** differential-expression, feature-selection, classification, imputation and batch-correction methods, not for biological or clinical conclusions.

* Registry page: https://registry.opendata.aws/oncogenerator-endo
* Code (MIT): https://github.com/NehaAr/OncoGenerator
* Data licence: CC BY 4.0
* Version: 1.1.0

More usage examples, tutorials and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).



### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

Each top-level prefix (a "folder") is one self-contained synthetic cohort:

| Prefix | What it is |
|---|---|
| `core_n1000/`, `core_n10000/`, `core_n100000/` | Core cohorts of 1,000 / 10,000 / 100,000 patients (seeds 1, 2, 3) |
| `scenario_noise_low_n10000/`, `scenario_noise_high_n10000/` | Same design, lower / higher measurement noise |
| `scenario_imbalanced_late10_n10000/` | Only ~10% of patients are late stage |
| `scenario_batch5_n10000/` | Five simulated batches with known batch effects |
| `scenario_missing_mcar10_n10000/`, `scenario_missing_mnar20_n10000/` | 10% missing completely at random / 20% missing not at random |

Inside every cohort prefix:

| File | Content |
|---|---|
| `clinical.csv.gz` | One row per patient: age, grade, stage, tumour type, BMI, ... (plus `Batch` in the batch scenario) |
| `proteins_log2fc.csv.gz` | Patient_ID plus 180 protein columns (log2 fold change, clipped to [-3, 3]; NaN = missing) |
| `ground_truth_proteins.csv.gz` | Per-protein generative parameters and pathway |
| `ground_truth_contrasts.csv.gz` | True effect of each protein for 4 contrasts (e.g. Late vs Early stage) |
| `run_metadata.json` | Generator version, seed, n, correlation, scenario parameters |

Extra files: `expected_log2fc_noise_free.csv.gz` (core 1k and 10k) and `ground_truth_batch_effects.csv.gz` (batch scenario). At the bucket root, `manifest.json` lists a SHA-256 for every file, and `data_dictionary.md`, `data_dictionary.json` and `README.md` document every column.

## Setup

This notebook uses only common tools: `boto3`, `pandas` and `matplotlib`.

In [ ]:
# Requirements (requirements.txt format):
# boto3>=1.34
# pandas>=2.0
# matplotlib>=3.7
# numpy>=1.24

import io
import json
from pprint import pprint

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from botocore import UNSIGNED
from botocore.config import Config

## Connecting to the bucket and listing the top-level prefixes

In [ ]:
BUCKET = 'oncogenerator-endo'  # Need toreplace with the final bucket name once the bucket is created
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

resp = s3.list_objects_v2(Bucket=BUCKET, Delimiter='/')
for p in resp.get('CommonPrefixes', []):
    print(p['Prefix'])
print('Root files:', [o['Key'] for o in resp.get('Contents', [])])

Looking inside one cohort prefix, we see the files described above: the clinical table, the protein table, the ground-truth files and the run metadata.

In [ ]:
# List the object keys within the 'core_n10000/' prefix
for item in s3.list_objects_v2(Bucket=BUCKET, Prefix='core_n10000/', MaxKeys=100)['Contents']:
    print(item['Key'])

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

All tables are **gzip-compressed CSV** (`.csv.gz`). CSV is readable by every language and tool, and the whole 10,000-patient cohort is only about 5 MB, so no special infrastructure is needed. Metadata are plain JSON and Markdown. The largest cohort (`core_n100000`) has a ~47 MB protein table.

Because the data are small and tabular, they work directly with pandas, R (`read.csv`) and DuckDB, and can be queried with Amazon Athena. A Parquet version can be generated with `scripts/build_release.py --format parquet` in the code repository.

Dataset-specific details: protein values are log2 fold changes clipped to [-3, 3], and an empty value (`NaN` once loaded) marks a missing measurement in the missing-data scenarios. Column definitions are in `data_dictionary.md` at the bucket root.

Next we define two helpers to read objects directly from S3 into memory.

In [ ]:
def read_csv_s3(key, **kw):
    body = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read()
    return pd.read_csv(io.BytesIO(body), compression='gzip', **kw)

def read_json_s3(key):
    return json.loads(s3.get_object(Bucket=BUCKET, Key=key)['Body'].read())

prefix = 'core_n10000'
meta = read_json_s3(f'{prefix}/run_metadata.json')
pprint(meta)

### Q: Can you show us an example of downloading and loading data from your dataset?

As an example, we load the 10,000-patient core cohort (`core_n10000`).

In [ ]:
clinical = read_csv_s3(f'{prefix}/clinical.csv.gz')
proteins = read_csv_s3(f'{prefix}/proteins_log2fc.csv.gz', index_col='Patient_ID')
truth = read_csv_s3(f'{prefix}/ground_truth_contrasts.csv.gz')

print(clinical.shape, proteins.shape, truth.shape)
clinical.head()

In [ ]:
proteins.iloc[:5, :8]

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

Left: how many patients fall into each stage. Right: average protein log2 fold change in early vs late stage patients (each dot is one of the 180 proteins).

In [ ]:
# Convert Late_Stage to True/False whether it is stored as 0/1, True/False or Yes/No
late = clinical.set_index('Patient_ID')['Late_Stage'].astype(str).str.lower().isin(['1', 'true', 'yes'])
late = late.reindex(proteins.index)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
clinical['Stage'].value_counts().sort_index().plot.bar(ax=ax[0], color='#4c72b0')
ax[0].set_title('Patients per stage (core_n10000)')
ax[0].set_ylabel('patients')

ax[1].scatter(proteins[~late].mean(), proteins[late].mean(), s=14, alpha=0.7)
lim = [-1.5, 1.5]
ax[1].plot(lim, lim, color='grey', lw=1)
ax[1].set_xlabel('mean log2FC, early stage')
ax[1].set_ylabel('mean log2FC, late stage')
ax[1].set_title('Protein means by stage group')
plt.tight_layout()
plt.show()

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Question: how well does a simple differential-expression test recover the true Late-vs-Early-stage effects?**

The generator records the true effect of every protein, so we can check an estimate against it. We estimate each protein's effect as the difference of group means and compare it with `true_effect_log2fc`.

In [ ]:
est = proteins[late].mean() - proteins[~late].mean()
tr = (truth[truth.contrast == 'Late_vs_Early_stage']
      .set_index('protein')['true_effect_log2fc'].reindex(est.index))

print('Pearson r (estimated vs true effect): %.3f' % np.corrcoef(est, tr)[0, 1])

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(tr, est, s=14, alpha=0.7)
lim = [min(tr.min(), est.min()), max(tr.max(), est.max())]
ax.plot(lim, lim, color='grey', lw=1)
ax.set_xlabel('true effect (log2FC)')
ax.set_ylabel('estimated effect (log2FC)')
ax.set_title('Late vs Early stage')
plt.show()

Compare the Pearson r printed above with 1: the closer it is, the better the simple estimate recovers the true effects. Use `is_large_effect` (|true effect| above the threshold in `run_metadata.json`) rather than `is_nonzero_effect` as the "positive" label for a detection benchmark: for some contrasts almost every protein has a tiny non-zero effect.

**Caveats.** The data are synthetic and must not be used for biological or clinical conclusions. Stage and grade scale protein effects by construction, `Survival_Outcome` is derived from stage and grade (leakage if you predict stage with it), and `Treatment`, `Myometrial_mm` and `Ethnicity` are noise columns.

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Which differential-expression or feature-selection methods keep their power and false-discovery control when the data are degraded?**

Compare the same method across `core_n10000`, `scenario_noise_high_n10000`, `scenario_imbalanced_late10_n10000`, `scenario_batch5_n10000`, `scenario_missing_mcar10_n10000` and `scenario_missing_mnar20_n10000`, scoring against `ground_truth_contrasts` (precision/recall on `is_large_effect`, error on `true_effect_log2fc`). Related challenges: batch correction scored against `ground_truth_batch_effects`, imputation scored against `expected_log2fc_noise_free`, and how sample size (1k vs 10k vs 100k) changes all of these.